# 📄 From Plain Text File to ChromaDB: Transparent Step-by-Step

In this notebook, we do **not** hide anything inside hardcoded lists.  
Everything starts from a real text file sitting directly in your workspace: **`company_knowledge_base.txt`**.

### 🎯 What We Will Learn Step-by-Step:
1. **Step 1: Read the Raw File**: Open `company_knowledge_base.txt` using standard Python.
2. **Step 2: Parse & Chunk the File**: Break down the long text into meaningful, distinct chunks with structured metadata (Company name, Section, Category).
3. **Step 3: Store in ChromaDB**: Create an explicit local database folder (`./my_local_chroma_db/`) and store our chunks.
4. **Step 4: Verify & Inspect Storage**: Query the database to inspect exactly what was saved (IDs, documents, and vector dimensions).
5. **Step 5: Semantic Retrieval**: Run natural language queries against the stored text chunks.

---
## Step 1: Read the Raw Text File

Let's verify that `company_knowledge_base.txt` exists and print out its contents.

In [1]:
import os

file_path = os.path.join(os.getcwd(), "company_knowledge_base.txt")

with open(file_path, "r", encoding="utf-8") as f:
    raw_content = f.read()

print(f" Loaded file from: {file_path}")
print(f" Total characters: {len(raw_content)}")
print("\n--- First 400 Characters of the File ---")
print(raw_content[:400])

 Loaded file from: c:\codeX\learn-AI\implementing_chromaDB\company_knowledge_base.txt
 Total characters: 7889

--- First 400 Characters of the File ---
COMPANY KNOWLEDGE BASE: ENTERPRISE PROFILES & PERFORMANCE (2024)

[COMPANY: Apex Robotics]
Overview:
Apex Robotics is a leading developer of autonomous mobile robots (AMRs) and industrial humanoid automation solutions designed for modern 


---
## Step 2: Chunking Strategy (Why & How)

> 🧠 **Concept: Why Chunk?**
> If you store the entire file as a single vector, its meaning becomes a blurry average of robotics, solar energy, cancer diagnostics, and finance.
> When someone asks *"What are Apex's risks?"*, the database would struggle to pinpoint the answer.
> 
> By chunking the file into individual company sections (Overview, Products, Financials, Partnerships, Risks), **each chunk gets its own crisp, precise vector**!

Let's parse each company block and split them by their sections:

In [2]:
import re

chunks = []
metadatas = []
ids = []

# Split by company delimiter
company_blocks = raw_content.split("--------------------------------------------------------------------------------")

chunk_counter = 1

for block in company_blocks:
    # Extract company name
    match_name = re.search(r"\[COMPANY:\s*(.*?)\]", block)
    if not match_name:
        continue
    
    company_name = match_name.group(1).strip()
    
    # Sections to split
    section_headers = ["Overview:", "Products & Technology:", "Financials & Growth:", "Partnerships & Future Outlook:", "Risk Factors & Challenges:"]
    
    # Extract each section
    for i, header in enumerate(section_headers):
        pattern = re.escape(header) + r"(.*?)(?=" + "|".join([re.escape(h) for h in section_headers[i+1:]] + ["$"]) + ")"
        section_match = re.search(pattern, block, re.DOTALL)
        
        if section_match:
            section_text = section_match.group(1).strip()
            clean_header = header.replace(":", "").strip()
            
            # Combine company name with section text for strong contextual embedding
            chunk_content = f"[{company_name} - {clean_header}]\n{section_text}"
            
            doc_id = f"chunk_{chunk_counter:03d}"
            metadata = {
                "source": "company_knowledge_base.txt",
                "company": company_name,
                "section": clean_header
            }
            
            chunks.append(chunk_content)
            metadatas.append(metadata)
            ids.append(doc_id)
            chunk_counter += 1

print(f" Created {len(chunks)} distinct chunks from the file!")
print("\n--- Sample Chunk (Chunk #1) ---")
print(f"ID: {ids[0]}")
print(f"Metadata: {metadatas[0]}")
print(f"Text:\n{chunks[0]}")

 Created 25 distinct chunks from the file!

--- Sample Chunk (Chunk #1) ---
ID: chunk_001
Metadata: {'source': 'company_knowledge_base.txt', 'company': 'Apex Robotics', 'section': 'Overview'}
Text:
[Apex Robotics - Overview]
Apex Robotics is a leading developer of autonomous mobile robots (AMRs) and industrial humanoid automation solutions designed for modern logistics hubs and manufacturing floors. Founded in 2017, the company is headquartered in Boston, Massachusetts.


---
## Step 3: Store into ChromaDB (Explicit Local Folder)

We now create an explicit local directory named **`./my_local_chroma_db`**.
You will see this folder directly in your file explorer!

In [3]:
import chromadb

# Explicit local folder path
db_dir = os.path.join(os.getcwd(), "my_local_chroma_db")

# Initialize ChromaDB Persistent Client
client = chromadb.PersistentClient(path=db_dir)

# Create or retrieve the collection
collection = client.get_or_create_collection(
    name="enterprise_profiles",
    metadata={"description": "Chunks from company_knowledge_base.txt"}
)

# Ingest all chunks using upsert (Update or Insert)
collection.upsert(
    ids=ids,
    documents=chunks,
    metadatas=metadatas
)

print(f" Successfully stored {collection.count()} chunks in ChromaDB!")
print(f"📁 Database location on disk: {db_dir}")

 Successfully stored 25 chunks in ChromaDB!
📁 Database location on disk: c:\codeX\learn-AI\implementing_chromaDB\my_local_chroma_db


---
## Step 4: Verify & Inspect What Is Stored

Let's inspect the database directly to confirm everything from the text file is stored safely.

In [4]:
# Retrieve 3 records with their embeddings
stored_data = collection.get(limit=3, include=["documents", "metadatas", "embeddings"])

print("🔍 Verifying stored records:")
for i in range(len(stored_data["ids"])):
    doc_id = stored_data["ids"][i]
    meta = stored_data["metadatas"][i]
    doc_text = stored_data["documents"][i]
    emb_dim = len(stored_data["embeddings"][i])
    
    print(f"\n  [Record {i+1}]")
    print(f"    ID:         {doc_id}")
    print(f"    Company:    {meta['company']}")
    print(f"    Section:    {meta['section']}")
    print(f"    Vector Dim: {emb_dim} float values")
    print(f"    Preview:    {doc_text[:100]}...")

🔍 Verifying stored records:

  [Record 1]
    ID:         chunk_001
    Company:    Apex Robotics
    Section:    Overview
    Vector Dim: 384 float values
    Preview:    [Apex Robotics - Overview]
Apex Robotics is a leading developer of autonomous mobile robots (AMRs) a...

  [Record 2]
    ID:         chunk_002
    Company:    Apex Robotics
    Section:    Products & Technology
    Vector Dim: 384 float values
    Preview:    [Apex Robotics - Products & Technology]
- Apex-X1: Autonomous heavy-load pallet mover capable of nav...

  [Record 3]
    ID:         chunk_003
    Company:    Apex Robotics
    Section:    Financials & Growth
    Vector Dim: 384 float values
    Preview:    [Apex Robotics - Financials & Growth]
In fiscal year 2024, Apex Robotics generated $1.82 billion in ...


---
## Step 5: Semantic Retrieval Testing

Now we test searching against our company text chunks!  
Notice how ChromaDB finds the exact matching company even without exact keywords.

In [5]:
# Query 1: Microcontroller supply chain issues in Taiwan
query1 = "Which company is facing supply chain bottlenecks from Taiwan?"
res1 = collection.query(query_texts=[query1], n_results=2)

print(f"🔎 Query: '{query1}'\n")
for i in range(len(res1["ids"][0])):
    print(f"Match #{i+1} (Distance: {res1['distances'][0][i]:.4f}):")
    print(f"Company: {res1['metadatas'][0][i]['company']} | Section: {res1['metadatas'][0][i]['section']}")
    print(f"{res1['documents'][0][i]}\n")

🔎 Query: 'Which company is facing supply chain bottlenecks from Taiwan?'

Match #1 (Distance: 1.1613):
Company: Apex Robotics | Section: Risk Factors & Challenges
[Apex Robotics - Risk Factors & Challenges]
Apex relies heavily on specialized semiconductor microcontrollers sourced from Taiwan. Supply chain bottlenecks and rising rare-earth magnet prices pose potential margin risks in the coming quarters.

Match #2 (Distance: 1.3441):
Company: FinEdge Capital | Section: Products & Technology
[FinEdge Capital - Products & Technology]
- AlphaPulse Engine: Ultra-low latency proprietary execution software processing microsecond-level order routing across 30+ global exchanges.
- DeepRisk Terminal: An institutional risk management dashboard analyzing portfolio value-at-risk (VaR) and systemic liquidity stress scenarios in real time.
- LiquidPools: Automated institutional market-making protocol providing liquidity for tokenized treasuries and cross-border settlement.



In [6]:
# Query 2: Early stage oncology and blood test kits
query2 = "Who develops liquid biopsy blood tests for cancer detection?"
res2 = collection.query(query_texts=[query2], n_results=2)

print(f"🔎 Query: '{query2}'\n")
for i in range(len(res2["ids"][0])):
    print(f"Match #{i+1} (Distance: {res2['distances'][0][i]:.4f}):")
    print(f"Company: {res2['metadatas'][0][i]['company']} | Section: {res2['metadatas'][0][i]['section']}")
    print(f"{res2['documents'][0][i]}\n")

🔎 Query: 'Who develops liquid biopsy blood tests for cancer detection?'

Match #1 (Distance: 0.6425):
Company: MediGen Diagnostics | Section: Products & Technology
[MediGen Diagnostics - Products & Technology]
- OncoScreen Ultra: A blood-based liquid biopsy test detecting circulating tumor DNA (ctDNA) for 12 distinct cancer types before symptom onset.
- PathoAI Suite: An FDA-cleared cloud imaging platform that aids pathologists in identifying micro-metastases in lymph node biopsies.
- GeneTrack Mobile: A point-of-care PCR device delivering rapid diagnostic results in under 25 minutes for infectious pathogens.

Match #2 (Distance: 0.9181):
Company: MediGen Diagnostics | Section: Overview
[MediGen Diagnostics - Overview]
MediGen Diagnostics is a clinical-stage biotechnology company based in San Diego, California. The company focuses on liquid biopsy diagnostic kits and multi-modal machine learning algorithms for early-stage oncology detection.



In [7]:
# Query 3: Filtered query (Only search inside 'CloudScale Systems')
query3 = "What is their annual revenue?"
res3 = collection.query(
    query_texts=[query3],
    n_results=1,
    where={"company": "CloudScale Systems"}
)

print(f"🔎 Query: '{query3}' (Filtered for CloudScale Systems)\n")
print(f"Company: {res3['metadatas'][0][0]['company']} | Section: {res3['metadatas'][0][0]['section']}")
print(f"{res3['documents'][0][0]}")

🔎 Query: 'What is their annual revenue?' (Filtered for CloudScale Systems)

Company: CloudScale Systems | Section: Financials & Growth
[CloudScale Systems - Financials & Growth]
Total revenue for 2024 reached $5.12 billion, representing a 48% increase year-over-year. CloudScale serves over 4,200 enterprise customers, including 42% of the Fortune 500. Operating cash flow stood at $1.65 billion.


---
## Summary

We took a raw human-readable text file (`company_knowledge_base.txt`), chunked it with metadata, saved it into an explicit local folder (`./my_local_chroma_db`), and performed semantic queries.

Next: We will look at **Chroma Online / Chroma Cloud** and how this exact same pipeline works over the internet!